In [ ]:
import zipfile
import xml.etree.ElementTree as ET
import os

namespaces = {
    'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'
}

def extract_content(docx_path):
    out = []
    with zipfile.ZipFile(docx_path) as docx:
        xml_content = docx.read('word/document.xml')
        root = ET.fromstring(xml_content)
        
        body = root.find('{' + namespaces['w'] + '}body')
        if body is None:
            return "No body found"
            
        for child in body:
            # Paragraph
            if child.tag == '{' + namespaces['w'] + '}p':
                p_text = []
                for run in child.iter('{' + namespaces['w'] + '}t'):
                    if run.text:
                        p_text.append(run.text)
                out.append("".join(p_text))
            # Table
            elif child.tag == '{' + namespaces['w'] + '}tbl':
                out.append("\n")
                first_row = True
                for row in child.iter('{' + namespaces['w'] + '}tr'):
                    row_cells = []
                    for cell in row.iter('{' + namespaces['w'] + '}tc'):
                        cell_text = []
                        for run in cell.iter('{' + namespaces['w'] + '}t'):
                            if run.text:
                                cell_text.append(run.text)
                        row_cells.append("".join(cell_text).replace("\n", " ").strip())
                    out.append("| " + " | ".join(row_cells) + " |")
                    if first_row:
                        out.append("|" + "---|"*len(row_cells))
                        first_row = False
                out.append("\n")
    return "\n".join(out)

docx_path = r"d:\flowsync\FlowSync_需求规格说明书.docx"
output_path = r"d:\flowsync\requirements.md"

try:
    content = extract_content(docx_path)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("Successfully wrote requirements to requirements.md!")
except Exception as e:
    print(f"Error: {e}")

In [2]:
import subprocess
import sys

# 1. 自动检测并安装 pymysql 驱动
try:
    import pymysql
except ImportError:
    print("正在为您安装连接 MySQL 所需的 pymysql 驱动库，请稍候...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pymysql"])
    import pymysql
    print("驱动库安装成功！")

# 2. 读取项目目录下的建表脚本
sql_file_path = r"d:\flowsync\flowsync-api\schema.sql"
print(f"正在读取 SQL 脚本: {sql_file_path}")
with open(sql_file_path, "r", encoding="utf-8") as f:
    sql_content = f.read()

# 3. 建立数据库连接 (默认使用 root / 123456 密码)
print("正在尝试连接本地 MySQL 数据库 (127.0.0.1:3306)...")
try:
    connection = pymysql.connect(
        host="127.0.0.1",
        user="root",
        password="",  # 若您本地密码不同，请在运行单元格前在此修改密码
        port=3306,
        autocommit=True
    )
    print("数据库连接成功！开始执行建库建表逻辑...")
except Exception as e:
    print(f"数据库连接失败: {e}")
    print("【请注意】：如果您的本地 MySQL 密码不是 123456，请双击本代码框修改 password 字段为您自己的密码后再试！")
    sys.exit(1)

# 4. 执行 SQL 脚本
cursor = connection.cursor()
# 按照分号拆分 SQL 语句，依次执行
statements = sql_content.split(";")
success_count = 0
error_count = 0

for statement in statements:
    stmt = statement.strip()
    if not stmt:
        continue
    # 剔除注释行
    lines = [line for line in stmt.split("\n") if not line.strip().startswith("--") and not line.strip().startswith("#")]
    clean_stmt = "\n".join(lines).strip()
    if not clean_stmt:
        continue
    
    try:
        cursor.execute(clean_stmt)
        success_count += 1
    except Exception as e:
        error_count += 1
        # 忽略已经存在报错以防止重复运行失败
        if "already exists" not in str(e).lower():
            print(f"执行 SQL 遇到错误: {e}\n语句: {clean_stmt[:100]}...")

print(f"\n🎉 导入完成！成功执行 {success_count} 条语句，异常 {error_count} 条（重复建表忽略异常为正常现象）。")
print("您现在可以直接到您的 VS Code 数据库客户端中点击 Refresh 刷新查看 flowsync_simple 数据库！")

cursor.close()
connection.close()

正在读取 SQL 脚本: d:\flowsync\flowsync-api\schema.sql
正在尝试连接本地 MySQL 数据库 (127.0.0.1:3306)...
数据库连接成功！开始执行建库建表逻辑...

🎉 导入完成！成功执行 12 条语句，异常 0 条（重复建表忽略异常为正常现象）。
您现在可以直接到您的 VS Code 数据库客户端中点击 Refresh 刷新查看 flowsync_simple 数据库！


In [1]:
import urllib.request
import json

# 填入您配置的 API Key 进行测试诊断
api_key = "sk-ws-H.EMLILRX.dwvB.MEUCIQD217G09HuLf81qTbgrL_1-zZZEfUW43mo7EpQVcHR8QgIgKYb02cHv1WYN7jj2kbKBtcOrO7mTA2sAZQ_uujZY2hU"
url = "https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

body = {
    "model": "qwen-plus",
    "messages": [
        {"role": "user", "content": "你好"}
    ]
}

print("正在直接发起 HTTP POST 请求连接通义千问 API (dashscope.aliyuncs.com)...")

req = urllib.request.Request(url, data=json.dumps(body).encode("utf-8"), headers=headers, method="POST")

try:
    with urllib.request.urlopen(req, timeout=10) as response:
        print("\n🟢 连接成功！")
        print("HTTP 状态码:", response.status)
        print("千问返回内容:", response.read().decode("utf-8"))
except Exception as e:
    print("\n🔴 连接失败！诊断信息如下:")
    if hasattr(e, 'code'):
        print("HTTP 状态码 (Status Code):", e.code)
    if hasattr(e, 'read'):
        try:
            error_msg = e.read().decode("utf-8")
            print("阿里云返回错误详情 (Error Detail):", error_msg)
        except Exception:
            pass
    else:
        print("网络连接异常:", e)


正在直接发起 HTTP POST 请求连接通义千问 API (dashscope.aliyuncs.com)...

🟢 连接成功！
HTTP 状态码: 200
千问返回内容: {"model":"qwen-plus","id":"chatcmpl-545415ff-12d7-99b1-a53a-14449adf0500","choices":[{"message":{"content":"你好！很高兴见到你～😊 有什么问题、想法，或者需要帮忙的地方吗？无论是学习、工作、生活中的小事，还是天马行空的创意，我都很乐意陪你一起探讨！✨","role":"assistant"},"index":0,"finish_reason":"stop"}],"created":1783496556,"object":"chat.completion","usage":{"total_tokens":53,"completion_tokens":44,"prompt_tokens":9,"prompt_tokens_details":{"cached_tokens":0}}}
